# Error-model accuracy: per-residue and per-protein MAE

Predicted vs observed |error|. 95% CIs from a protein-cluster bootstrap.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

PRED_PATH = Path("../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json")
N_BOOT = 2000
WITHIN_PPM = 0.2        # "per-protein prediction within X ppm" threshold
WITHIN_REL = 0.2        # "per-protein prediction within X relative error" threshold
SEED = 0

## 1. Load valid residues

In [ ]:
d = json.load(open(PRED_PATH))
rows = []
for pid in d["pred"]:
    p = np.asarray(d["pred"][pid], float)
    g = np.asarray(d["oof_resid"][pid], float)
    m = np.asarray(d["mask"][pid], bool) & np.asarray(d["oof_mask"][pid], bool)
    n = min(len(p), len(g), len(m))
    valid = m[:n] & np.isfinite(p[:n]) & np.isfinite(g[:n])
    for i in np.where(valid)[0]:
        rows.append((pid, int(i) + 1, p[i], g[i]))
res = pd.DataFrame(rows, columns=["entry", "resnum", "pred", "obs"])
prot = res.groupby("entry").agg(pred=("pred", "mean"), obs=("obs", "mean"), n=("obs", "size"))
print(f"{len(res)} residues, {len(prot)} proteins")
print(f"observed |error|: mean {res.obs.mean():.3f}, median {res.obs.median():.3f} ppm | "
      f"predicted: mean {res.pred.mean():.3f}, median {res.pred.median():.3f} ppm")

## 2. Point estimates

In [ ]:
def metrics(pred, obs):
    err = pred - obs
    return dict(MAE=np.abs(err).mean(), medAE=np.median(np.abs(err)), RMSE=np.sqrt((err ** 2).mean()),
                bias=err.mean())


summary = []
for level, df in [("per residue", res), ("per protein", prot)]:
    m = metrics(df.pred.to_numpy(), df.obs.to_numpy())
    summary.append(dict(level=level, n=len(df), **m,
                        pearson=pearsonr(df.pred, df.obs)[0], spearman=spearmanr(df.pred, df.obs)[0]))
summary = pd.DataFrame(summary).set_index("level")
pd.set_option("display.width", 220)
print(summary.round(3).to_string())

# per-protein "within" fractions
a_model = np.abs(prot.pred - prot.obs)
print(f"\nper protein, within {WITHIN_PPM} ppm      : {100 * (a_model < WITHIN_PPM).mean():.0f}%")
print(f"per protein, within {100 * WITHIN_REL:.0f}% relative : {100 * (a_model / prot.obs < WITHIN_REL).mean():.0f}%")
print(f"per protein, median relative error: {100 * np.median(a_model / prot.obs):.1f}%")


## 3. Protein-cluster bootstrap 95% CIs

In [ ]:
rng = np.random.default_rng(SEED)
entries = prot.index.to_numpy()
res_by_entry = {e: g for e, g in res.groupby("entry")}

boot = {k: [] for k in ["res_MAE", "prot_MAE"]}
for _ in range(N_BOOT):
    pick = rng.choice(entries, len(entries), replace=True)
    r = pd.concat([res_by_entry[e] for e in pick])
    p = prot.loc[pick]
    for tag, df in [("res", r), ("prot", p)]:
        boot[f"{tag}_MAE"].append(np.abs(df.pred - df.obs).mean())

ci = {k: np.percentile(v, [2.5, 97.5]) for k, v in boot.items()}
for tag, label in [("res", "per residue"), ("prot", "per protein")]:
    print(f"{label:12s} MAE {summary.loc[label, 'MAE']:.3f} [{ci[f'{tag}_MAE'][0]:.3f}, {ci[f'{tag}_MAE'][1]:.3f}] ppm")


## 4. Sensitivity: per-protein MAE without the under-predicted outliers

In [ ]:
b1, b0 = np.polyfit(prot.pred, prot.obs, 1)
resid = prot.obs - (b1 * prot.pred + b0)
q1, q3 = np.percentile(resid, [25, 75])
under = resid > q3 + 1.5 * (q3 - q1)
kept = prot[~under]
print(f"under-predicted outliers removed: {int(under.sum())} -> {len(kept)} proteins")
print(f"per-protein MAE {np.abs(kept.pred - kept.obs).mean():.3f} ppm, "
      f"Pearson {pearsonr(kept.pred, kept.obs)[0]:.3f}, Spearman {spearmanr(kept.pred, kept.obs)[0]:.3f}")
print("removed:", sorted(prot.index[under]))